In [ ]:
# ── 0A  Install dependencies ────────────────────────────────────────────
import subprocess
import sys
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

# Torch — uncomment the line that matches your hardware:
pip('torch','torchvision','--index-url','https://download.pytorch.org/whl/rocm6.4')  # AMD ROCm 6.4
# pip('torch','torchvision','--index-url','https://download.pytorch.org/whl/rocm6.2')  # AMD ROCm 6.2
# pip('torch','torchvision','--index-url','https://download.pytorch.org/whl/cu124')    # NVIDIA CUDA 12.4

# diffusers from source (ZImagePipeline not yet on PyPI)
pip('git+https://github.com/huggingface/diffusers')

# core runtime
pip('safetensors>=0.7.0','accelerate>=1.0.0','transformers>=5.0.0',
    'peft>=0.18.0','huggingface_hub>=1.9.0',
    'datasets>=3.0.0','pandas>=2.0.0',
    'matplotlib>=3.9.0','ipywidgets>=8.0.0',
    'optimum-quanto>=0.2.0','Pillow>=10.0.0')

print('All packages installed.')

# support CJK characters in matplotlib
import os
import urllib.request

font_url = "https://github.com/googlefonts/noto-cjk/raw/main/Sans/OTF/SimplifiedChinese/NotoSansCJKsc-Regular.otf"
font_path = "NotoSansCJKsc-Regular.otf"

if not os.path.exists(font_path):
    urllib.request.urlretrieve(font_url, font_path)

from matplotlib import font_manager, rcParams
font_manager.fontManager.addfont(font_path)

prop = font_manager.FontProperties(fname=font_path)
font_name = prop.get_name()

rcParams["font.family"] = font_name
rcParams["axes.unicode_minus"] = False

In [ ]:
# ── 0B  HF Auth + Download model + dataset in one shot ──────────────────
import os
from huggingface_hub import login, snapshot_download, hf_hub_download
import torch

# MANUALLY ADD YOUR TOKEN HERE
HF_TOKEN = '__'  # <- Replace with your real token
login(token=HF_TOKEN)
os.environ['HF_TOKEN'] = HF_TOKEN

MODEL_REPO   = 'DownFlow/Z-Image-Turbo-Fuli'
DATASET_REPO = 'DownFlow/fuliji'
PARQUET_FILE = 'dataset.parquet'

print(f'[1/1] Downloading {MODEL_REPO}  (~20 GB, cached on repeat runs) ...')
MODEL_DIR = snapshot_download(MODEL_REPO, token=HF_TOKEN)
print(f'  -> {MODEL_DIR}')

# GPU check
if torch.cuda.is_available():
    d = torch.cuda.get_device_properties(0)
    print(f'GPU: {d.name}  {d.total_memory/2**30:.1f} GB VRAM')
else:
    print('WARNING: no CUDA GPU detected.')
print('Setup complete.')

In [ ]:
# ── 1A  Load pipeline ────────────────────────────────────────────────────
import torch
from diffusers import ZImagePipeline

print(f'Loading {MODEL_DIR} ...')
pipe = ZImagePipeline.from_pretrained(
    MODEL_DIR, torch_dtype=torch.bfloat16, low_cpu_mem_usage=False,
).to('cuda')
pipe.set_progress_bar_config(disable=False)
print('Pipeline ready.')

In [ ]:
# ── 1B  Prompt  (edit and re-run freely) ─────────────────────────────────
PROMPT   = 'Diane Kruger is on a bed in a large bedroom, she is crawling on a bed near the edge naked, legs apart showing a shaved vagina, asshole and bum, she has beautiful legs and feet, high detail, 8k'  # <- edit me
NEG      = 'blurry, low quality, deformed, watermark, text'
STEPS    = 25
CFG      = 4.0
SEED     = 1337
N_IMAGES = 6 # number of samples to generate side-by-side

import matplotlib.pyplot as plt
import os
from datetime import datetime

g = torch.Generator('cuda').manual_seed(SEED)
with torch.no_grad():
    out = pipe(
        prompt=[PROMPT] * N_IMAGES,
        negative_prompt=[NEG] * N_IMAGES,
        height=704, width=1280,
        num_inference_steps=STEPS,
        guidance_scale=CFG,
        generator=g,
    )

# Display the images
fig, axes = plt.subplots(1, N_IMAGES, figsize=(4 * N_IMAGES, 4.2))
if N_IMAGES == 1: axes = [axes]
fig.suptitle(PROMPT[:100], fontsize=11, y=1.01)  # Truncate long prompt for title
for ax, img in zip(axes, out.images):
    ax.imshow(img)
    ax.axis('off')
plt.tight_layout()
plt.show()

# ── Save generated image(s) to disk ─────────────────────────────────────
output_dir = "generated_images"
os.makedirs(output_dir, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
for idx, img in enumerate(out.images):
    filename = f"{timestamp}_seed{SEED}_{idx+1}.png"
    filepath = os.path.join(output_dir, filename)
    img.save(filepath)
    print(f"✅ Saved: {filepath}")